# Paired Wilcoxon Test for User Perception Study

This notebook performs:

- Paired **Wilcoxon signed-rank test**
- **Effect size** \(r = Z / \sqrt{N}\)

for:

- **Keras** (`safe_mode=False` vs `safe_mode=True`)
- **PyTorch** (`weights_only=False` vs `weights_only=True`)

> Filtered to participants who reported having loaded/shared ML models at least once.


In [ ]:
!pip install pandas numpy scipy

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import wilcoxon

In [2]:
csv_path = "Survey - Model Sharing (Responses) - Form Responses.csv"
df = pd.read_csv(csv_path)
df.head(10)

,Timestamp,Which of the following best describes your area of expertise?,How would you rate your level of expertise in the field of Machine Learning?\n,Have you ever loaded a pre-trained machine learning model onto your system or computer that was shared by someone else or downloaded from the internet?,"Given this snippet of code, how comfortable would you feel when loading the shared model?","If you feel that the previous situation raises any concerns, which one(s)?","Given this snippet of code, how comfortable would you feel when loading the shared model? .1","If you feel that the previous situation raises any concerns, which one(s)?.1","Given this snippet of code, how comfortable would you feel when loading the shared model? .2","If you feel that the previous situation raises any concerns, which one(s)?.2","Given this snippet of code, how comfortable would you feel when loading the shared model? \n\nNote: This question also includes the process of importing the model definition or code provided by the same source from which you downloaded the model weights.","If you feel that the previous situation raises any concerns, which one(s)?.3","Have you ever inspected a shared model file before loading it?\n\nThis means performing actions such as unzipping the file (if required by the format), understanding the files it contains, reviewing their contents, and evaluating the internals of the model before loading it into your system.","Knowing that platforms like Hugging Face provide security tools for their repositories, including malware scanning, pickle scanning, secrets scanning, and third-party AI model scanners such as JFrog and Protect AI—if the shared model you were to load was downloaded from such platforms, would your perception of comfort in doing so change?","If you feel that the previous situation raises any concerns, which one(s)?.4"
0,7/4/2025 17:55:52,Machine Learning / Artificial Intelligence,4,Yes,9,Challenging to ensure ethical AI compliance,10,Challenging to ensure ethical AI compliance,10,Challenging to ensure ethical AI compliance,7,Manipulated training process,No,I would feel more comfortable loading the model,Challenging to ensure ethical AI compliance
1,7/4/2025 18:01:31,Machine Learning / Artificial Intelligence,4,Yes,4,Arbitrary code execution,8,NaN,5,Arbitrary code execution,9,NaN,Yes,I would feel more comfortable loading the model,NaN
2,7/4/2025 18:02:26,Cybersecurity,3,Yes,2,Arbitrary code execution,7,"Poor performance, Manipulated training process",3,Arbitrary code execution,3,Arbitrary code execution,Yes,I would feel more comfortable loading the model,"Manipulated training process, Licensing, Chall..."
3,7/4/2025 18:02:32,HPC and FPGA acceleration,2,Yes,5,Arbitrary code execution,10,NaN,8,Arbitrary code execution,10,NaN,No,I would feel more comfortable loading the model,NaN
4,7/4/2025 18:02:37,Machine Learning / Artificial Intelligence,3,Yes,8,"Arbitrary code execution, Challenging to ensur...",9,"Poor performance, Licensing",8,"Arbitrary code execution, Challenging to ensur...",9,"Poor performance, Licensing",No,It would not affect my perception,"Manipulated training process, Licensing, Chall..."
5,7/4/2025 18:12:40,Machine Learning / Artificial Intelligence,4,Yes,8,Source of the model,8,source of the model,6,May not load correctly under different hardwar...,9,NaN,Yes,I would feel more comfortable loading the model,NaN
6,7/4/2025 18:13:27,Software Engineering,3,Yes,3,Arbitrary code execution,10,NaN,10,NaN,3,Arbitrary code execution,No,I would feel more comfortable loading the model,NaN
7,7/4/2025 18:18:10,Cybersecurity,4,Yes,3,"Manipulated training process, Arbitrary code e...",5,Manipulated training process,7,"Poor performance, Manipulated training process",7,"Poor performance, Manipulated training process",No,I would feel more comfortable loading the model,"Poor performance, Manipulated training process"
8,7/4/2025 18:26:57,Software Engineering,2,No,3,"Poor performance, Manipulated training process.


Keep only participants who reported having loaded/shared ML models.


In [3]:
df = df[df["Have you ever loaded a pre-trained machine learning model onto your system or computer that was shared by someone else or downloaded from the internet?"] == "Yes"]
N = len(df)
print(f"Filtered participants: {N}")

Filtered participants: 53


Select the answers to include in the test.

In [4]:
keras_false = df["Given this snippet of code, how comfortable would you feel when loading the shared model? "].astype(float)
keras_true  = df["Given this snippet of code, how comfortable would you feel when loading the shared model? .1"].astype(float)

torch_false = df["Given this snippet of code, how comfortable would you feel when loading the shared model? .2"].astype(float)
torch_true  = df["Given this snippet of code, how comfortable would you feel when loading the shared model? \n\nNote: This question also includes the process of importing the model definition or code provided by the same source from which you downloaded the model weights."].astype(float)



Define Test Function


In [ ]:
def run_test(a, b, label):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)

    # Pairwise drop NaNs
    mask = ~np.isnan(a) & ~np.isnan(b)
    a, b = a[mask], b[mask]

    # Drop 0-differences to extract number n of non-zero pairs
    d = b - a
    nz = d != 0
    a2, _ = a[nz], b[nz]
    n = len(a2)

    w = wilcoxon(a, b, zero_method="wilcox", method="asymptotic")

    p = w.pvalue
    z = w.zstatistic
    r = z / np.sqrt(n)

    print(f"\n--- {label} ---")
    print(f"N_eff (non-zero pairs): {n}")
    print(f"p-value: {p}")
    print(f"Effect size r: {abs(r)}  (report |r|)")


Run the statistical test

In [8]:
run_test(keras_false, keras_true, "Keras safe_mode")
run_test(torch_false, torch_true, "PyTorch weights_only")

NameError: name 'n' is not defined

## Interpretation

- **p-value**: the probability of observing results at least as extreme as those obtained assuming the null hypothesis is true. A p-value below 0.05 (or another chosen significance level) indicates that the observed difference is statistically significant and the null hypothesis can be rejected.
- **r**: magnitude of perception shift.
